# Drosophila Brain Cocaine Response — Single-Cell RNA-Seq Analysis


## Step 0: Project root, imports, figure/results directories

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\bmp\sc_project2_drosophila_brain")
DATA_DIR = PROJECT_ROOT / "data"

RESULT_DIR = PROJECT_ROOT / "results"
FIG_DIR = RESULT_DIR / "figures"
TABLES_DIR = RESULT_DIR / "tables"

for d in (RESULT_DIR, FIG_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"data/ not found at {DATA_DIR} - check PROJECT_ROOT"

import sys
import gzip
import shutil
import gc
import math

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sc.settings.figdir = str(FIG_DIR)

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT.resolve())
print("Data dir:", DATA_DIR.resolve())
print("Figures dir:", FIG_DIR.resolve())
print("Results dir:", RESULT_DIR.resolve())
print("Tables dir:", TABLES_DIR.resolve())

## Step 1: Fix `features.tsv.gz` files (space-delimited -> tab-delimited)
Idempotent — safe to rerun; skips files already fixed.

In [ ]:
sample_dirs = sorted([
    p for p in DATA_DIR.iterdir()
    if p.is_dir() and (p / "matrix.mtx.gz").exists()
])
print(f"Found {len(sample_dirs)} samples: {[p.name for p in sample_dirs]}")

def features_needs_fix(path):
    with gzip.open(path, "rt") as f:
        first_line = f.readline()
    return "\t" not in first_line

def fix_features_file(path):
    backup = path.with_suffix(path.suffix + ".bak")
    if not backup.exists():
        shutil.copy(path, backup)

    with gzip.open(path, "rt") as f:
        lines = f.readlines()

    fixed_lines = []
    for line in lines:
        line = line.rstrip("\n")
        parts = line.split(" ", 2)  # gene_id, gene_symbol, feature_type ("Gene Expression")
        fixed_lines.append("\t".join(parts))

    with gzip.open(path, "wt") as f:
        f.write("\n".join(fixed_lines) + "\n")

for sample_path in sample_dirs:
    feat_file = sample_path / "features.tsv.gz"
    if features_needs_fix(feat_file):
        fix_features_file(feat_file)
        print(f"Fixed: {feat_file}")
    else:
        print(f"Already OK: {feat_file}")

## Step 2: Load all 8 samples into one AnnData object

In [ ]:
adatas = {}
for sample_path in sample_dirs:
    sample_name = sample_path.name
    adata_sample = sc.read_10x_mtx(
        path=sample_path, var_names="gene_symbols", cache=False
    )
    parts = sample_name.split("_")  # e.g. "Female_Cocaine_1"
    sex, treatment, replicate = parts[0], parts[1], parts[2]

    adata_sample.obs["sample"] = sample_name
    adata_sample.obs["sex"] = sex
    adata_sample.obs["treatment"] = treatment
    adata_sample.obs["replicate"] = replicate
    adata_sample.obs["condition"] = f"{sex}_{treatment}"

    adatas[sample_name] = adata_sample

adata = ad.concat(adatas, label="sample_batch", index_unique="-", join="outer")
adata.var_names_make_unique()

print(f"Merged Dataset: {adata.n_obs} cells x {adata.n_vars} genes")

## Step 3: QC metrics & visualization

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("mt:")
adata.var["ribo"] = adata.var_names.str.startswith(("RpS", "RpL", "rps", "rpl"))

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
    groupby="condition",
    jitter=0.4,
    rotation=45,
    multi_panel=True,
    save="_QC_violin.png"
)

## Step 4: Filter cells & genes

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs["pct_counts_mt"] < 10.0, :].copy()

print(f"Cells remaining after QC: {adata.n_obs}")

## Step 5: Drop genes with missing/invalid names

In [ ]:
valid_genes = adata.var_names.notna() & (adata.var_names != "")
print(f"Dropping {(~valid_genes).sum()} genes with missing/invalid names")

adata = adata[:, valid_genes].copy()
adata.var_names_make_unique()

print(f"Genes remaining: {adata.n_vars}")

## Step 6: Stratified subsample (documented memory-constraint limitation)


In [ ]:
target_per_sample = 4000  # lower this if the memory wall is hit again

print(adata.obs["sample"].value_counts())

sampled_idx = []
for sample_name, group in adata.obs.groupby("sample"):
    n = min(target_per_sample, len(group))
    sampled_idx.extend(group.sample(n=n, random_state=RANDOM_STATE).index)

adata = adata[sampled_idx].copy()

for col in ["sample", "sex", "treatment", "replicate", "condition"]:
    if hasattr(adata.obs[col], "cat"):
        adata.obs[col] = adata.obs[col].cat.remove_unused_categories()

print(f"Subsampled dataset: {adata.n_obs} cells x {adata.n_vars} genes")
print(adata.obs["sample"].value_counts())

## Step 7: Normalize, log-transform, select highly variable genes

In [ ]:
adata.X = adata.X.astype("float32")

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata  # full-gene, log-normalized data kept for later marker/DE lookups

sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")
sc.pl.highly_variable_genes(adata, save="_HVG.png")

assert "highly_variable" in adata.var.columns
print(f"{adata.var['highly_variable'].sum()} HVGs selected")

## Step 8: Subset to HVGs, scale, PCA

In [ ]:
adata = adata[:, adata.var["highly_variable"]].copy()
gc.collect()
print(f"Subset to HVGs: {adata.n_obs} cells x {adata.n_vars} genes")

sc.pp.scale(adata, max_value=10)
adata.X = adata.X.astype("float32")
gc.collect()

sc.tl.pca(adata, n_comps=30, svd_solver="arpack", use_highly_variable=True, random_state=RANDOM_STATE)
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=30, save="_PCA_variance.png")

## Checkpoint: save preprocessed, PCA-ready AnnData for the next notebook

In [ ]:
checkpoint_path = RESULT_DIR / "checkpoint_01_preprocessed.h5ad"
adata.write_h5ad(checkpoint_path)
print("Saved checkpoint:", checkpoint_path.resolve())
print(f"{adata.n_obs} cells x {adata.n_vars} genes (HVG-subset, scaled, PCA computed)")